# SIH26168 — Step 3: Coordinate Systems & IMU Alignment

In this notebook, we establish the coordinate transformation pipeline required by the SIH problem statement:
1. **Phone Body Frame** ($X_{phone}, Y_{phone}, Z_{phone}$) to **Vehicle Body Frame** ($X_{forward}, Y_{lateral}, Z_{up}$).
2. **Stationary Gravity Alignment**: Leveling roll and pitch angles using the static gravity reaction vector.
3. **Dynamic Yaw Alignment**: Estimating forward azimuth from longitudinal acceleration bursts.
4. **Geodetic (WGS84) to Local ENU (East-North-Up)**: Converting lat/lon to Cartesian meters for linear dead reckoning and Kalman filtering.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data.loader import load_trip
from src.preprocessing.gravity_alignment import align_phone_to_vehicle
from src.preprocessing.orientation import geodetic_to_enu

In [ ]:
# Load Trip Vta02
df_phone, df_veh = load_trip('Vta02')
raw_accel = df_phone[['accel_x', 'accel_y', 'accel_z']].values
raw_gyro = df_phone[['gyro_x', 'gyro_y', 'gyro_z']].values
speed = df_veh['veh_speed_ms'].values

# Perform Automatic Phone-to-Vehicle Alignment
accel_veh, gyro_veh, R_pv, angles = align_phone_to_vehicle(raw_accel, raw_gyro, speed)
print("=== Computed Alignment Matrix R_pv ===")
print(np.round(R_pv, 4))
print("\nEstimated Mounting Offsets:")
for k, v in angles.items():
    print(f"  {k}: {v:.2f}°")

In [ ]:
# Convert Geodetic Trajectory to Local ENU (meters)
lat0 = df_veh['veh_lat'].iloc[0]
lon0 = df_veh['veh_lon'].iloc[0]

east, north, up = geodetic_to_enu(df_veh['veh_lat'].values, df_veh['veh_lon'].values, lat0, lon0)
df_veh['east_m'] = east
df_veh['north_m'] = north

plt.figure(figsize=(9, 7))
plt.plot(east, north, 'b-', lw=2, label='Ground Truth ENU Path')
plt.plot(0, 0, 'go', markersize=10, label='Origin (0,0)')
plt.xlabel('East (meters)')
plt.ylabel('North (meters)')
plt.title('Vehicle Trajectory in Local Cartesian ENU Frame (meters)', fontweight='bold')
plt.grid(True)
plt.axis('equal')
plt.legend()
plt.show()